# 方向予測と相場環境の診断

**Historical archive / 過去の研究記録**

原本のコードを保持しています。独立実行や現在の検証基準への適合は保証しません。前のセルの変数に依存する箇所があります。実行入口は `../08_trade_quality.ipynb` を参照してください。

保存出力は `../../results/legacy/`、既知の問題は `../../docs/AUDIT.md` に整理しています。


## 元Notebookのセル 10

出典: `FX.ipynb`、0始まりのindex=9。コード内容は変更していません。

In [ ]:
# ============================================================
# 方向AI 診断コード
# 目的:
# 1. 方向予測が本当に有効か確認
# 2. BUY / SELLを別々に評価
# 3. 確率が高いほど本当に当たりやすいか確認
# 4. 相場レジーム別に性能を見る
# 5. Walk-Forwardで壊れる期間を特定
# ============================================================


# ============================================================
# 1. 追加ライブラリ
# ============================================================

from sklearn.metrics import (
    roc_auc_score,
    brier_score_loss
)


# ============================================================
# 2. 方向AI単体のテスト対象を作る
# ============================================================

# 方向AIは「実際にMOVEしたケースだけ」で評価する
# なぜならAI②は
# 「動くなら上か下か」
# を学習しているから

direction_test = test[
    test["move_target"] == 1
].copy()


# テスト対象が空でないことを確認
print(
    "方向AI評価対象数:",
    len(direction_test)
)


# ============================================================
# 3. 方向AIの確率予測
# ============================================================

direction_test_prob = (
    direction_model
    .predict_proba(
        direction_test[
            direction_features
        ]
    )
)


direction_class_map = {
    c: i
    for i, c
    in enumerate(
        direction_model.classes_
    )
}


direction_test[
    "p_down"
] = (
    direction_test_prob[
        :,
        direction_class_map[0]
    ]
)


direction_test[
    "p_up"
] = (
    direction_test_prob[
        :,
        direction_class_map[1]
    ]
)


# ============================================================
# 4. 方向AIのROC-AUC
# ============================================================

direction_auc = roc_auc_score(
    direction_test[
        "direction_target"
    ],
    direction_test[
        "p_up"
    ]
)


print("\n==============================")
print("方向AI 単体性能")
print("==============================")

print(
    "Direction ROC-AUC:",
    round(
        direction_auc,
        4
    )
)


# ============================================================
# 5. Brier Score
# ============================================================

# Brier Scoreは
# 「予測確率がどれくらい現実に近いか」
# を見る指標
#
# 0に近いほど良い

brier = brier_score_loss(
    direction_test[
        "direction_target"
    ],
    direction_test[
        "p_up"
    ]
)

print(
    "Brier Score:",
    round(
        brier,
        4
    )
)


# ============================================================
# 6. 方向予測クラス
# ============================================================

direction_test[
    "direction_pred"
] = np.where(
    direction_test[
        "p_up"
    ] >= 0.5,
    1,
    0
)


direction_accuracy = (
    direction_test[
        "direction_pred"
    ]
    ==
    direction_test[
        "direction_target"
    ]
).mean()


print(
    "Direction Accuracy:",
    round(
        direction_accuracy * 100,
        2
    ),
    "%"
)


# ============================================================
# 7. 予測確率帯ごとの実績
# ============================================================

bins = [
    0.00,
    0.40,
    0.45,
    0.50,
    0.55,
    0.60,
    0.65,
    0.70,
    0.75,
    1.00
]


labels = [
    "0-40%",
    "40-45%",
    "45-50%",
    "50-55%",
    "55-60%",
    "60-65%",
    "65-70%",
    "70-75%",
    "75-100%"
]


direction_test[
    "prob_group"
] = pd.cut(
    direction_test[
        "p_up"
    ],
    bins=bins,
    labels=labels,
    include_lowest=True
)


probability_table = (
    direction_test
    .groupby(
        "prob_group",
        observed=False
    )
    .agg(
        count=(
            "direction_target",
            "size"
        ),
        actual_up_rate=(
            "direction_target",
            "mean"
        ),
        avg_future_return=(
            "future_return",
            "mean"
        )
    )
)


probability_table[
    "actual_up_rate"
] *= 100


probability_table[
    "avg_future_return"
] *= 100


print("\n==============================")
print("予測確率帯ごとの実績")
print("==============================")

print(
    probability_table
)


# ============================================================
# 8. BUY側の診断
# ============================================================

buy_thresholds = [
    0.55,
    0.60,
    0.65,
    0.70,
    0.75
]


buy_analysis = []


for threshold in buy_thresholds:

    subset = direction_test[
        direction_test[
            "p_up"
        ] >= threshold
    ]


    if len(subset) == 0:
        continue


    buy_win_rate = (
        subset[
            "future_return"
        ] > 0
    ).mean()


    buy_avg_return = (
        subset[
            "future_return"
        ].mean()
    )


    buy_analysis.append({
        "threshold":
            threshold,

        "trades":
            len(subset),

        "win_rate":
            buy_win_rate,

        "average_return":
            buy_avg_return
    })


buy_df = pd.DataFrame(
    buy_analysis
)


if len(buy_df) > 0:

    buy_df[
        "win_rate"
    ] *= 100

    buy_df[
        "average_return"
    ] *= 100


print("\n==============================")
print("BUY確率別分析")
print("==============================")

print(
    buy_df
)


# ============================================================
# 9. SELL側の診断
# ============================================================

sell_thresholds = [
    0.55,
    0.60,
    0.65,
    0.70,
    0.75
]


sell_analysis = []


for threshold in sell_thresholds:

    subset = direction_test[
        direction_test[
            "p_down"
        ] >= threshold
    ]


    if len(subset) == 0:
        continue


    sell_win_rate = (
        subset[
            "future_return"
        ] < 0
    ).mean()


    sell_avg_return = (
        -subset[
            "future_return"
        ]
    ).mean()


    sell_analysis.append({
        "threshold":
            threshold,

        "trades":
            len(subset),

        "win_rate":
            sell_win_rate,

        "average_return":
            sell_avg_return
    })


sell_df = pd.DataFrame(
    sell_analysis
)


if len(sell_df) > 0:

    sell_df[
        "win_rate"
    ] *= 100

    sell_df[
        "average_return"
    ] *= 100


print("\n==============================")
print("SELL確率別分析")
print("==============================")

print(
    sell_df
)


# ============================================================
# 10. レジーム判定用特徴
# ============================================================

# -----------------------------
# ボラティリティ
# -----------------------------

vol_median = (
    direction_test[
        "volatility_1h"
    ].median()
)


direction_test[
    "vol_regime"
] = np.where(
    direction_test[
        "volatility_1h"
    ] >= vol_median,
    "HIGH_VOL",
    "LOW_VOL"
)


# -----------------------------
# トレンド / レンジ
# -----------------------------

# MA50傾きの絶対値が中央値以上なら
# TREND
# それ以下ならRANGE

slope_abs = abs(
    direction_test[
        "MA50_slope"
    ]
)


slope_median = (
    slope_abs.median()
)


direction_test[
    "trend_regime"
] = np.where(
    slope_abs
    >= slope_median,
    "TREND",
    "RANGE"
)


# -----------------------------
# 上昇トレンド / 下落トレンド
# -----------------------------

direction_test[
    "trend_direction"
] = np.where(
    direction_test[
        "MA50_slope"
    ] > 0,
    "UP_TREND",
    "DOWN_TREND"
)


# ============================================================
# 11. レジーム別性能関数
# ============================================================

def evaluate_regime(
    df_regime,
    name
):

    if len(df_regime) < 20:
        return None


    try:

        auc = roc_auc_score(
            df_regime[
                "direction_target"
            ],
            df_regime[
                "p_up"
            ]
        )

    except ValueError:

        auc = np.nan


    accuracy = (
        (
            (
                df_regime[
                    "p_up"
                ] >= 0.5
            ).astype(int)
            ==
            df_regime[
                "direction_target"
            ]
        )
        .mean()
    )


    strong_buy = df_regime[
        df_regime[
            "p_up"
        ] >= 0.60
    ]


    if len(strong_buy) > 0:

        buy_win = (
            strong_buy[
                "future_return"
            ] > 0
        ).mean()

        buy_avg = (
            strong_buy[
                "future_return"
            ].mean()
        )

    else:

        buy_win = np.nan
        buy_avg = np.nan


    strong_sell = df_regime[
        df_regime[
            "p_down"
        ] >= 0.60
    ]


    if len(strong_sell) > 0:

        sell_win = (
            strong_sell[
                "future_return"
            ] < 0
        ).mean()

        sell_avg = (
            -strong_sell[
                "future_return"
            ]
        ).mean()

    else:

        sell_win = np.nan
        sell_avg = np.nan


    return {

        "regime":
            name,

        "samples":
            len(df_regime),

        "auc":
            auc,

        "accuracy":
            accuracy,

        "buy_trades":
            len(strong_buy),

        "buy_win_rate":
            buy_win,

        "buy_avg_return":
            buy_avg,

        "sell_trades":
            len(strong_sell),

        "sell_win_rate":
            sell_win,

        "sell_avg_return":
            sell_avg
    }


# ============================================================
# 12. レジーム別集計
# ============================================================

regime_results = []


# HIGH / LOW VOL
for regime_name in [
    "HIGH_VOL",
    "LOW_VOL"
]:

    subset = direction_test[
        direction_test[
            "vol_regime"
        ] == regime_name
    ]

    result = evaluate_regime(
        subset,
        regime_name
    )

    if result is not None:
        regime_results.append(
            result
        )


# TREND / RANGE
for regime_name in [
    "TREND",
    "RANGE"
]:

    subset = direction_test[
        direction_test[
            "trend_regime"
        ] == regime_name
    ]

    result = evaluate_regime(
        subset,
        regime_name
    )

    if result is not None:
        regime_results.append(
            result
        )


# UP / DOWN TREND
for regime_name in [
    "UP_TREND",
    "DOWN_TREND"
]:

    subset = direction_test[
        direction_test[
            "trend_direction"
        ] == regime_name
    ]

    result = evaluate_regime(
        subset,
        regime_name
    )

    if result is not None:
        regime_results.append(
            result
        )


regime_df = pd.DataFrame(
    regime_results
)


if len(regime_df) > 0:

    regime_df[
        "accuracy"
    ] *= 100

    regime_df[
        "buy_win_rate"
    ] *= 100

    regime_df[
        "sell_win_rate"
    ] *= 100

    regime_df[
        "buy_avg_return"
    ] *= 100

    regime_df[
        "sell_avg_return"
    ] *= 100


print("\n==============================")
print("レジーム別方向AI性能")
print("==============================")

print(
    regime_df
)


# ============================================================
# 13. 確率校正を見る
# ============================================================

# 10分割して
# モデルの平均予測確率
# 実際の上昇率
# を比較する

direction_test[
    "calibration_bin"
] = pd.qcut(
    direction_test[
        "p_up"
    ],
    q=10,
    duplicates="drop"
)


calibration_table = (
    direction_test
    .groupby(
        "calibration_bin",
        observed=False
    )
    .agg(
        count=(
            "direction_target",
            "size"
        ),
        mean_predicted_prob=(
            "p_up",
            "mean"
        ),
        actual_up_rate=(
            "direction_target",
            "mean"
        )
    )
)


calibration_table[
    "mean_predicted_prob"
] *= 100

calibration_table[
    "actual_up_rate"
] *= 100


print("\n==============================")
print("確率校正")
print("==============================")

print(
    calibration_table
)


# ============================================================
# 14. 確率校正グラフ
# ============================================================

plt.figure(
    figsize=(8, 6)
)


plt.plot(
    calibration_table[
        "mean_predicted_prob"
    ],
    calibration_table[
        "actual_up_rate"
    ],
    marker="o"
)


plt.plot(
    [0, 100],
    [0, 100],
    linestyle="--"
)


plt.xlabel(
    "Predicted UP Probability (%)"
)

plt.ylabel(
    "Actual UP Rate (%)"
)

plt.title(
    "Direction Probability Calibration"
)

plt.grid()

plt.show()


# ============================================================
# 15. Walk-Forward方向AI診断
# ============================================================

print("\n==============================")
print("Walk-Forward Direction Diagnosis")
print("==============================")


N_SPLITS_DIAG = 5

block_size_diag = (
    len(data)
    // (N_SPLITS_DIAG + 1)
)


wf_direction_results = []


for fold in range(
    N_SPLITS_DIAG
):


    train_end = (
        block_size_diag
        * (fold + 1)
    )


    test_start = (
        train_end
        + GAP
    )


    test_end = (
        test_start
        + block_size_diag
    )


    if test_end > len(data):

        test_end = len(data)


    train_fold = (
        data.iloc[
            :train_end
        ]
    )


    test_fold = (
        data.iloc[
            test_start:test_end
        ]
    )


    # MOVEしたケースだけ学習
    direction_train_fold = (
        train_fold[
            train_fold[
                "move_target"
            ] == 1
        ]
    )


    # MOVEしたケースだけ方向AIを評価
    direction_test_fold = (
        test_fold[
            test_fold[
                "move_target"
            ] == 1
        ]
    )


    if (
        len(
            direction_train_fold
        ) < 100
        or
        len(
            direction_test_fold
        ) < 20
    ):

        continue


    fold_model = RandomForestClassifier(

        n_estimators=400,

        max_depth=8,

        min_samples_leaf=15,

        max_features="sqrt",

        class_weight="balanced",

        random_state=42,

        n_jobs=-1
    )


    fold_model.fit(

        direction_train_fold[
            direction_features
        ],

        direction_train_fold[
            "direction_target"
        ]
    )


    fold_prob = (
        fold_model
        .predict_proba(
            direction_test_fold[
                direction_features
            ]
        )
    )


    fold_map = {

        c: i

        for i, c

        in enumerate(
            fold_model.classes_
        )
    }


    fold_p_down = (
        fold_prob[
            :,
            fold_map[0]
        ]
    )


    fold_p_up = (
        fold_prob[
            :,
            fold_map[1]
        ]
    )


    try:

        fold_auc = roc_auc_score(
            direction_test_fold[
                "direction_target"
            ],
            fold_p_up
        )

    except ValueError:

        fold_auc = np.nan


    fold_pred = (
        fold_p_up >= 0.5
    ).astype(int)


    fold_accuracy = (
        fold_pred
        ==
        direction_test_fold[
            "direction_target"
        ].values
    ).mean()


    # BUY 60%以上
    buy_mask = (
        fold_p_up >= 0.60
    )


    if buy_mask.sum() > 0:

        buy_returns = (
            direction_test_fold[
                "future_return"
            ].values[
                buy_mask
            ]
        )

        buy_win = (
            buy_returns > 0
        ).mean()

        buy_avg = (
            buy_returns.mean()
        )

    else:

        buy_win = np.nan
        buy_avg = np.nan


    # SELL 60%以上
    sell_mask = (
        fold_p_down >= 0.60
    )


    if sell_mask.sum() > 0:

        sell_returns = (
            direction_test_fold[
                "future_return"
            ].values[
                sell_mask
            ]
        )

        sell_win = (
            sell_returns < 0
        ).mean()

        sell_avg = (
            -sell_returns
        ).mean()

    else:

        sell_win = np.nan
        sell_avg = np.nan


    wf_direction_results.append({

        "fold":
            fold + 1,

        "train_samples":
            len(
                direction_train_fold
            ),

        "test_samples":
            len(
                direction_test_fold
            ),

        "auc":
            fold_auc,

        "accuracy":
            fold_accuracy,

        "buy_trades":
            buy_mask.sum(),

        "buy_win_rate":
            buy_win,

        "buy_avg_return":
            buy_avg,

        "sell_trades":
            sell_mask.sum(),

        "sell_win_rate":
            sell_win,

        "sell_avg_return":
            sell_avg
    })


wf_direction_df = pd.DataFrame(
    wf_direction_results
)


if len(
    wf_direction_df
) > 0:

    wf_direction_df[
        "accuracy"
    ] *= 100

    wf_direction_df[
        "buy_win_rate"
    ] *= 100

    wf_direction_df[
        "sell_win_rate"
    ] *= 100

    wf_direction_df[
        "buy_avg_return"
    ] *= 100

    wf_direction_df[
        "sell_avg_return"
    ] *= 100


print(
    wf_direction_df
)


# ============================================================
# 16. Walk-Forward平均
# ============================================================

print("\n==============================")
print("方向AI Walk-Forward平均")
print("==============================")


if len(
    wf_direction_df
) > 0:

    print(
        "平均AUC:",
        round(
            wf_direction_df[
                "auc"
            ].mean(),
            4
        )
    )

    print(
        "平均Accuracy:",
        round(
            wf_direction_df[
                "accuracy"
            ].mean(),
            2
        ),
        "%"
    )

    print(
        "平均BUY勝率:",
        round(
            wf_direction_df[
                "buy_win_rate"
            ].mean(),
            2
        ),
        "%"
    )

    print(
        "平均BUYリターン:",
        round(
            wf_direction_df[
                "buy_avg_return"
            ].mean(),
            4
        ),
        "%"
    )

    print(
        "平均SELL勝率:",
        round(
            wf_direction_df[
                "sell_win_rate"
            ].mean(),
            2
        ),
        "%"
    )

    print(
        "平均SELLリターン:",
        round(
            wf_direction_df[
                "sell_avg_return"
            ].mean(),
            4
        ),
        "%"
    )


# ============================================================
# 17. AUCをグラフ化
# ============================================================

if len(
    wf_direction_df
) > 0:

    plt.figure(
        figsize=(10, 5)
    )

    plt.plot(
        wf_direction_df[
            "fold"
        ],
        wf_direction_df[
            "auc"
        ],
        marker="o"
    )

    plt.axhline(
        0.5,
        linestyle="--"
    )

    plt.xlabel(
        "Fold"
    )

    plt.ylabel(
        "Direction ROC-AUC"
    )

    plt.title(
        "Walk-Forward Direction AUC"
    )

    plt.grid()

    plt.show()

## 元Notebookのセル 11

出典: `FX.ipynb`、0始まりのindex=10。コード内容は変更していません。

In [ ]:
# ============================================================
# 勝てる相場 / 負ける相場を特定する診断コード
# ============================================================


# ============================================================
# 1. ATRを作る
# ============================================================

# True Range
tr1 = df["High"] - df["Low"]
tr2 = abs(df["High"] - df["Close"].shift(1))
tr3 = abs(df["Low"] - df["Close"].shift(1))

true_range = pd.concat(
    [tr1, tr2, tr3],
    axis=1
).max(axis=1)

# 14本ATR
df["ATR14"] = (
    true_range
    .rolling(14)
    .mean()
)

# 価格で割って比率化
df["ATR14_pct"] = (
    df["ATR14"]
    / df["Close"]
)


# ============================================================
# 2. ADXを作る
# ============================================================

# 上方向の動き
plus_dm = (
    df["High"]
    - df["High"].shift(1)
)

# 下方向の動き
minus_dm = (
    df["Low"].shift(1)
    - df["Low"]
)

plus_dm = np.where(
    (plus_dm > minus_dm)
    & (plus_dm > 0),
    plus_dm,
    0.0
)

minus_dm = np.where(
    (minus_dm > plus_dm)
    & (minus_dm > 0),
    minus_dm,
    0.0
)

plus_dm = pd.Series(
    plus_dm,
    index=df.index
)

minus_dm = pd.Series(
    minus_dm,
    index=df.index
)

atr_adx = (
    true_range
    .rolling(14)
    .mean()
)

plus_di = (
    100
    * plus_dm
    .rolling(14)
    .mean()
    / atr_adx
)

minus_di = (
    100
    * minus_dm
    .rolling(14)
    .mean()
    / atr_adx
)

dx = (
    100
    * abs(plus_di - minus_di)
    / (plus_di + minus_di)
)

df["ADX14"] = (
    dx
    .rolling(14)
    .mean()
)


# ============================================================
# 3. data側にもATR / ADXを追加
# ============================================================

# indexを使ってdfから取得
data["ATR14_pct"] = (
    df.loc[
        data.index,
        "ATR14_pct"
    ]
)

data["ADX14"] = (
    df.loc[
        data.index,
        "ADX14"
    ]
)

# NaN除去
data = (
    data
    .dropna(
        subset=[
            "ATR14_pct",
            "ADX14"
        ]
    )
    .copy()
)


# ============================================================
# 4. レジーム判定
# ============================================================

# ボラティリティ中央値
vol_threshold = (
    data[
        "volatility_1h"
    ].median()
)

# MA50 slope絶対値中央値
slope_threshold = (
    abs(
        data[
            "MA50_slope"
        ]
    ).median()
)

# ADX閾値
# 一般的に20〜25以上で
# トレンドがあると見ることが多い
ADX_THRESHOLD = 25


data["high_vol_regime"] = (
    data["volatility_1h"]
    >= vol_threshold
).astype(int)


data["ma_trend_regime"] = (
    abs(
        data["MA50_slope"]
    )
    >= slope_threshold
).astype(int)


data["adx_trend_regime"] = (
    data["ADX14"]
    >= ADX_THRESHOLD
).astype(int)


# ============================================================
# 5. 時間帯カテゴリ
# ============================================================

# 日本時間ベース
hours = data.index.hour


def classify_session(hour):

    # 東京時間
    if 8 <= hour < 15:
        return "TOKYO"

    # 欧州
    elif 15 <= hour < 22:
        return "LONDON"

    # NY
    elif 22 <= hour or hour < 5:
        return "NEW_YORK"

    else:
        return "OTHER"


data["session"] = [
    classify_session(h)
    for h in hours
]


# ============================================================
# 6. Walk-ForwardでFoldごとの特徴を計測
# ============================================================

N_SPLITS_ANALYSIS = 5

block_size_analysis = (
    len(data)
    // (N_SPLITS_ANALYSIS + 1)
)

fold_analysis = []


for fold in range(
    N_SPLITS_ANALYSIS
):

    train_end = (
        block_size_analysis
        * (fold + 1)
    )

    test_start = (
        train_end
        + GAP
    )

    test_end = (
        test_start
        + block_size_analysis
    )

    if test_end > len(data):
        test_end = len(data)


    train_fold = (
        data.iloc[
            :train_end
        ]
    )

    test_fold = (
        data.iloc[
            test_start:test_end
        ]
    )


    if (
        len(train_fold) < 100
        or
        len(test_fold) < 100
    ):
        continue


    # ----------------------------------------
    # AI① MOVE
    # ----------------------------------------

    move_model_fold = RandomForestClassifier(

        n_estimators=300,

        max_depth=8,

        min_samples_leaf=20,

        max_features="sqrt",

        class_weight="balanced",

        random_state=42,

        n_jobs=-1
    )


    move_model_fold.fit(

        train_fold[
            move_features
        ],

        train_fold[
            "move_target"
        ]
    )


    move_prob_fold = (
        move_model_fold
        .predict_proba(
            test_fold[
                move_features
            ]
        )[:, 1]
    )


    # ----------------------------------------
    # AI② Direction
    # ----------------------------------------

    direction_train_fold = (
        train_fold[
            train_fold[
                "move_target"
            ] == 1
        ]
    )

    direction_test_fold = (
        test_fold[
            test_fold[
                "move_target"
            ] == 1
        ]
    )


    direction_model_fold = (
        RandomForestClassifier(

            n_estimators=300,

            max_depth=8,

            min_samples_leaf=15,

            max_features="sqrt",

            class_weight="balanced",

            random_state=42,

            n_jobs=-1
        )
    )


    direction_model_fold.fit(

        direction_train_fold[
            direction_features
        ],

        direction_train_fold[
            "direction_target"
        ]
    )


    # 方向AIのAUC
    if len(direction_test_fold) > 20:

        direction_prob_fold = (
            direction_model_fold
            .predict_proba(
                direction_test_fold[
                    direction_features
                ]
            )
        )

        class_map_fold = {
            c: i
            for i, c
            in enumerate(
                direction_model_fold.classes_
            )
        }

        p_up_direction = (
            direction_prob_fold[
                :,
                class_map_fold[1]
            ]
        )

        try:

            direction_auc_fold = (
                roc_auc_score(
                    direction_test_fold[
                        "direction_target"
                    ],
                    p_up_direction
                )
            )

        except ValueError:

            direction_auc_fold = np.nan

    else:

        direction_auc_fold = np.nan


    # ----------------------------------------
    # 全テストに方向確率
    # ----------------------------------------

    direction_prob_all = (
        direction_model_fold
        .predict_proba(
            test_fold[
                direction_features
            ]
        )
    )

    class_map_all = {
        c: i
        for i, c
        in enumerate(
            direction_model_fold.classes_
        )
    }

    p_down_all = (
        direction_prob_all[
            :,
            class_map_all[0]
        ]
    )

    p_up_all = (
        direction_prob_all[
            :,
            class_map_all[1]
        ]
    )


    # ----------------------------------------
    # 売買シグナル
    # ----------------------------------------

    signals = np.zeros(
        len(test_fold)
    )


    buy_mask = (

        (move_prob_fold >= MOVE_PROB_THRESHOLD)

        &

        (p_up_all >= DIRECTION_PROB_THRESHOLD)

        &

        (
            p_up_all
            - p_down_all
            >= DIRECTION_MARGIN
        )
    )


    sell_mask = (

        (move_prob_fold >= MOVE_PROB_THRESHOLD)

        &

        (p_down_all >= DIRECTION_PROB_THRESHOLD)

        &

        (
            p_down_all
            - p_up_all
            >= DIRECTION_MARGIN
        )
    )


    signals[
        buy_mask
    ] = 1

    signals[
        sell_mask
    ] = -1


    # ----------------------------------------
    # 非重複バックテスト
    # ----------------------------------------

    trade_returns = []

    buy_returns = []

    sell_returns = []

    i = 0


    while i < len(
        test_fold
    ):

        signal = (
            signals[i]
        )

        if signal == 0:

            i += 1
            continue


        row = (
            test_fold.iloc[i]
        )


        if signal == 1:

            r = (
                row[
                    "future_return"
                ]
                - TRADING_COST
            )

            buy_returns.append(
                r
            )


        else:

            r = (
                -row[
                    "future_return"
                ]
                - TRADING_COST
            )

            sell_returns.append(
                r
            )


        trade_returns.append(
            r
        )


        i += HOLD_BARS


    trade_returns = np.array(
        trade_returns
    )

    buy_returns = np.array(
        buy_returns
    )

    sell_returns = np.array(
        sell_returns
    )


    # ----------------------------------------
    # Fold損益
    # ----------------------------------------

    if len(
        trade_returns
    ) > 0:

        fold_win_rate = (
            trade_returns > 0
        ).mean()

        fold_avg_return = (
            trade_returns.mean()
        )

    else:

        fold_win_rate = np.nan
        fold_avg_return = np.nan


    # ----------------------------------------
    # 相場環境
    # ----------------------------------------

    avg_volatility = (
        test_fold[
            "volatility_1h"
        ].mean()
    )

    avg_atr = (
        test_fold[
            "ATR14_pct"
        ].mean()
    )

    avg_adx = (
        test_fold[
            "ADX14"
        ].mean()
    )

    avg_ma50_slope = (
        test_fold[
            "MA50_slope"
        ].mean()
    )

    trend_rate = (
        test_fold[
            "ma_trend_regime"
        ].mean()
    )

    adx_trend_rate = (
        test_fold[
            "adx_trend_regime"
        ].mean()
    )

    high_vol_rate = (
        test_fold[
            "high_vol_regime"
        ].mean()
    )


    # ----------------------------------------
    # UP / DOWN割合
    # ----------------------------------------

    move_only = (
        test_fold[
            test_fold[
                "move_target"
            ] == 1
        ]
    )


    if len(move_only) > 0:

        up_rate = (
            move_only[
                "direction_target"
            ].mean()
        )

        down_rate = (
            1
            - up_rate
        )

    else:

        up_rate = np.nan
        down_rate = np.nan


    # ----------------------------------------
    # 時間帯割合
    # ----------------------------------------

    session_counts = (
        test_fold[
            "session"
        ]
        .value_counts(
            normalize=True
        )
    )


    tokyo_rate = (
        session_counts
        .get(
            "TOKYO",
            0
        )
    )

    london_rate = (
        session_counts
        .get(
            "LONDON",
            0
        )
    )

    ny_rate = (
        session_counts
        .get(
            "NEW_YORK",
            0
        )
    )


    # ----------------------------------------
    # BUY / SELL集計
    # ----------------------------------------

    buy_trade_count = len(
        buy_returns
    )

    sell_trade_count = len(
        sell_returns
    )


    if buy_trade_count > 0:

        buy_win_rate = (
            buy_returns > 0
        ).mean()

        buy_avg_return = (
            buy_returns.mean()
        )

    else:

        buy_win_rate = np.nan
        buy_avg_return = np.nan


    if sell_trade_count > 0:

        sell_win_rate = (
            sell_returns > 0
        ).mean()

        sell_avg_return = (
            sell_returns.mean()
        )

    else:

        sell_win_rate = np.nan
        sell_avg_return = np.nan


    # ----------------------------------------
    # 期間
    # ----------------------------------------

    fold_start = (
        test_fold.index.min()
    )

    fold_end = (
        test_fold.index.max()
    )


    # ----------------------------------------
    # 保存
    # ----------------------------------------

    fold_analysis.append({

        "fold":
            fold + 1,

        "start":
            fold_start,

        "end":
            fold_end,

        "samples":
            len(
                test_fold
            ),

        "direction_auc":
            direction_auc_fold,

        "trades":
            len(
                trade_returns
            ),

        "win_rate":
            fold_win_rate,

        "avg_return":
            fold_avg_return,

        "buy_trades":
            buy_trade_count,

        "buy_win_rate":
            buy_win_rate,

        "buy_avg_return":
            buy_avg_return,

        "sell_trades":
            sell_trade_count,

        "sell_win_rate":
            sell_win_rate,

        "sell_avg_return":
            sell_avg_return,

        "avg_volatility":
            avg_volatility,

        "avg_ATR":
            avg_atr,

        "avg_ADX":
            avg_adx,

        "avg_MA50_slope":
            avg_ma50_slope,

        "trend_rate":
            trend_rate,

        "adx_trend_rate":
            adx_trend_rate,

        "high_vol_rate":
            high_vol_rate,

        "up_rate":
            up_rate,

        "down_rate":
            down_rate,

        "tokyo_rate":
            tokyo_rate,

        "london_rate":
            london_rate,

        "ny_rate":
            ny_rate
    })


# ============================================================
# 7. DataFrame化
# ============================================================

fold_df = pd.DataFrame(
    fold_analysis
)


# ============================================================
# 8. %表示
# ============================================================

percent_columns = [

    "win_rate",
    "avg_return",

    "buy_win_rate",
    "buy_avg_return",

    "sell_win_rate",
    "sell_avg_return",

    "trend_rate",
    "adx_trend_rate",
    "high_vol_rate",

    "up_rate",
    "down_rate",

    "tokyo_rate",
    "london_rate",
    "ny_rate"
]


for col in percent_columns:

    if col in fold_df.columns:

        fold_df[col] *= 100


# ============================================================
# 9. Fold一覧
# ============================================================

print(
    "\n=============================="
)

print(
    "Fold別 相場診断"
)

print(
    "=============================="
)

print(
    fold_df[
        [
            "fold",
            "start",
            "end",

            "direction_auc",

            "trades",
            "win_rate",
            "avg_return",

            "avg_volatility",
            "avg_ATR",
            "avg_ADX",
            "avg_MA50_slope",

            "trend_rate",
            "adx_trend_rate",
            "high_vol_rate",

            "up_rate",
            "down_rate"
        ]
    ]
)


# ============================================================
# 10. BUY / SELL診断
# ============================================================

print(
    "\n=============================="
)

print(
    "BUY / SELL Fold別"
)

print(
    "=============================="
)

print(
    fold_df[
        [
            "fold",

            "buy_trades",
            "buy_win_rate",
            "buy_avg_return",

            "sell_trades",
            "sell_win_rate",
            "sell_avg_return"
        ]
    ]
)


# ============================================================
# 11. 時間帯診断
# ============================================================

print(
    "\n=============================="
)

print(
    "時間帯割合"
)

print(
    "=============================="
)

print(
    fold_df[
        [
            "fold",
            "tokyo_rate",
            "london_rate",
            "ny_rate"
        ]
    ]
)


# ============================================================
# 12. 勝ちFold / 負けFold
# ============================================================

winning_folds = (
    fold_df[
        fold_df[
            "avg_return"
        ] > 0
    ]
)

losing_folds = (
    fold_df[
        fold_df[
            "avg_return"
        ] <= 0
    ]
)


# ============================================================
# 13. 勝ちFold平均
# ============================================================

compare_columns = [

    "direction_auc",

    "avg_volatility",
    "avg_ATR",
    "avg_ADX",
    "avg_MA50_slope",

    "trend_rate",
    "adx_trend_rate",
    "high_vol_rate",

    "up_rate",
    "down_rate",

    "tokyo_rate",
    "london_rate",
    "ny_rate"
]


print(
    "\n=============================="
)

print(
    "勝ちFold平均"
)

print(
    "=============================="
)

if len(
    winning_folds
) > 0:

    print(
        winning_folds[
            compare_columns
        ]
        .mean()
    )


# ============================================================
# 14. 負けFold平均
# ============================================================

print(
    "\n=============================="
)

print(
    "負けFold平均"
)

print(
    "=============================="
)

if len(
    losing_folds
) > 0:

    print(
        losing_folds[
            compare_columns
        ]
        .mean()
    )


# ============================================================
# 15. 差を見る
# ============================================================

if (
    len(
        winning_folds
    ) > 0
    and
    len(
        losing_folds
    ) > 0
):

    difference = (

        winning_folds[
            compare_columns
        ]
        .mean()

        -

        losing_folds[
            compare_columns
        ]
        .mean()
    )


    difference = (
        difference
        .sort_values(
            ascending=False
        )
    )


    print(
        "\n=============================="
    )

    print(
        "勝ちFold - 負けFold の差"
    )

    print(
        "=============================="
    )

    print(
        difference
    )


# ============================================================
# 16. AUCと平均リターン
# ============================================================

plt.figure(
    figsize=(10, 5)
)

plt.scatter(
    fold_df[
        "direction_auc"
    ],
    fold_df[
        "avg_return"
    ]
)

plt.xlabel(
    "Direction AUC"
)

plt.ylabel(
    "Average Return (%)"
)

plt.title(
    "Direction AUC vs Strategy Return"
)

plt.grid()

plt.show()


# ============================================================
# 17. ADXと平均リターン
# ============================================================

plt.figure(
    figsize=(10, 5)
)

plt.scatter(
    fold_df[
        "avg_ADX"
    ],
    fold_df[
        "avg_return"
    ]
)

plt.xlabel(
    "Average ADX"
)

plt.ylabel(
    "Average Return (%)"
)

plt.title(
    "ADX vs Strategy Return"
)

plt.grid()

plt.show()


# ============================================================
# 18. ボラティリティと平均リターン
# ============================================================

plt.figure(
    figsize=(10, 5)
)

plt.scatter(
    fold_df[
        "avg_volatility"
    ],
    fold_df[
        "avg_return"
    ]
)

plt.xlabel(
    "Average Volatility"
)

plt.ylabel(
    "Average Return (%)"
)

plt.title(
    "Volatility vs Strategy Return"
)

plt.grid()

plt.show()


# ============================================================
# 19. Trend率と平均リターン
# ============================================================

plt.figure(
    figsize=(10, 5)
)

plt.scatter(
    fold_df[
        "trend_rate"
    ],
    fold_df[
        "avg_return"
    ]
)

plt.xlabel(
    "Trend Regime Rate (%)"
)

plt.ylabel(
    "Average Return (%)"
)

plt.title(
    "Trend Rate vs Strategy Return"
)

plt.grid()

plt.show()


# ============================================================
# 20. 相関
# ============================================================

correlation_columns = [

    "avg_return",

    "direction_auc",

    "avg_volatility",
    "avg_ATR",
    "avg_ADX",
    "avg_MA50_slope",

    "trend_rate",
    "adx_trend_rate",
    "high_vol_rate",

    "up_rate",
    "down_rate",

    "tokyo_rate",
    "london_rate",
    "ny_rate"
]


correlation_table = (
    fold_df[
        correlation_columns
    ]
    .corr()[
        "avg_return"
    ]
    .sort_values(
        ascending=False
    )
)


print(
    "\n=============================="
)

print(
    "平均リターンとの相関"
)

print(
    "=============================="
)

print(
    correlation_table
)